In [ ]:
import os
import pandas as pd
import numpy as np

REPO_DIR = os.path.join("/Users/haya1/Documents/LanguageModel_Labels/headlines_prediction")
# REPO_DIR = "."
os.chdir(REPO_DIR)
data_dir = os.path.join(REPO_DIR, "Data")
temp_dir = os.path.join(REPO_DIR, "Temp")

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def euclidean_distance(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.linalg.norm(a - b)


In [44]:
def decode_embed_responses(responses):
    responses = responses.reset_index(drop=True)
    responses_decoded = []
    for _, response in responses.iterrows(): 
        response_out =  response.response['body']['data'][0]
        responses_decoded.append({
            "custom_id": response["custom_id"],
            "Embeddings": response_out['embedding'],
            "Tokens": int(response.response["body"]["usage"]["prompt_tokens"])
        })
    return(pd.json_normalize(responses_decoded))

In [45]:
responses_headline_clean = pd.read_json(os.path.join(temp_dir, 'Embeddings/Responses/responses_headline_clean_part1.jsonl'), lines=True)
headline_embeddings = decode_embed_responses(responses_headline_clean)

headline_embeddings.rename(columns={
    'custom_id': 'headline_id',
    'Embeddings': 'headline_clean_embed',
    'Tokens': 'DescriptionCleanTokens'
    }, inplace=True)

print(len(headline_embeddings))

10000


In [46]:
N = len(headline_embeddings)
samples = headline_embeddings['headline_clean_embed'].sample(n=int(2*N), replace=True, random_state=123).reset_index(drop=True)

random_pairs = pd.DataFrame({
    "i": samples[:N].reset_index(drop=True),
    "j": samples[N:].reset_index(drop=True)
    }) 

rand_cosine = random_pairs.apply(lambda x: cosine_similarity(x["i"], x["j"]), axis=1).mean()
rand_euclidean = random_pairs.apply(lambda x: euclidean_distance(x["i"], x["j"]), axis=1).mean()

random_baseline = pd.DataFrame({
    "Metric": ["CosineSimilarity", "EuclideanDistance"],
    "RandomBaseline": [rand_cosine, rand_euclidean]
    }) 
print(random_baseline)

random_baseline_path = os.path.join(data_dir, "random_baseline.csv")
random_baseline.to_csv(random_baseline_path, index=False)
print(f"Saved {os.path.basename(random_baseline_path)}, n = {len(random_baseline)}, at {os.path.dirname(random_baseline_path)}")

              Metric  RandomBaseline
0   CosineSimilarity        0.308662
1  EuclideanDistance        1.171722
Saved random_baseline.csv, n = 2, at /Users/haya1/Documents/LanguageModel_Labels/headlines_completion/Data


In [47]:
responses_headline_llm_clean_part1 = pd.read_json(os.path.join(temp_dir, 'Embeddings/Responses/responses_headline_llm_clean_part1.jsonl'), lines=True)
responses_headline_llm_clean_part2 = pd.read_json(os.path.join(temp_dir, 'Embeddings/Responses/responses_headline_llm_clean_part2.jsonl'), lines=True)
responses_headline_llm = pd.concat([responses_headline_llm_clean_part1, responses_headline_llm_clean_part2])
print(len(responses_headline_llm))

description_llm_embeddings = decode_embed_responses(responses_headline_llm)

description_llm_embeddings.rename(columns={
    'custom_id': 'id',
    'Embeddings': 'headline_llm_clean_embed',
    'Tokens': 'DescriptionLLMCleanTokens'
    }, inplace=True)

headlines_completion = pd.read_csv(os.path.join(data_dir, f"headlines_completion.csv"))
headlines_completion = headlines_completion.merge(headline_embeddings, on="headline_id").merge(description_llm_embeddings, on="id")

headlines_completion["TextSimilarity"] = headlines_completion["headline_clean"] == headlines_completion["headline_llm_clean"]
headlines_completion["EuclideanDistance"] = headlines_completion.apply(lambda x: euclidean_distance(x["headline_clean_embed"], x["headline_llm_clean_embed"]), axis=1)
headlines_completion["CosineSimilarity"] = headlines_completion.apply(lambda x: cosine_similarity(x["headline_clean_embed"], x["headline_llm_clean_embed"]), axis=1)

59997


In [48]:
headline_completion_similarity = headlines_completion[['headline_id', 'prompt_template_id', 'response_format', 'add_date',
    'model', 'temperature', 'max_tokens', 'headline_trim', 'headline_llm',
    'input_tokens', 'output_tokens', 'date', 'headline',
    'company_name', 'headline_clean', 'headline_llm_clean', 'id',
    'TextSimilarity', 'EuclideanDistance', 'CosineSimilarity']]
headline_completion_similarity_path = os.path.join(data_dir, "headlines_completion_similarity.csv")
headline_completion_similarity.to_csv(headline_completion_similarity_path, index=False)
print(f"Saved {os.path.basename(headline_completion_similarity_path)}, n = {len(headline_completion_similarity)}, at {os.path.dirname(headline_completion_similarity_path)}")

Saved headlines_completion_similarity.csv, n = 59997, at /Users/haya1/Documents/LanguageModel_Labels/headlines_completion/Data
